# NB2 - Container Open-Vocabulary (YOLO-World)

Fine-tunes YOLO-World on the container dataset, runs the eight-prompt zero-shot
sweep on the held-out test split, applies the pre-registered T2 verdict, and
pushes `ausdm-v3-container-openvocab` for NB5.

**Kaggle setup:** phone-verified account, internet on, Secret `ROBOFLOW_API_KEY`.
Rebuilds the container data from Roboflow with the same seed-0 split as NB0/NB1,
so it is self-contained.

**Runtime ~1.5 to 1.9 h GPU** (fine-tune ~1.4 h, prompt sweep ~10 to 20 min,
rest a few min). Use Save Version - Save and Run All (Commit).

This is the notebook that carries the paper's central claim: whether zero-shot
open-vocabulary detection fails on container text, and whether a prompt recovers it.


## 0. One-time environment setup

In [ ]:
# Same pins as NB0/NB1. Uninstall ray to avoid the broken raytune callback.
! pip install -q ultralytics==8.3.40 roboflow==1.1.50 pillow==10.4.0
! pip uninstall -y -q ray 2>/dev/null || true

## 1. Imports, constants, seeds

In [ ]:
# Standard library
import json, os, sys, shutil, subprocess, time, random
from pathlib import Path

# Block ultralytics 8.3.40's broken raytune callback
sys.modules['ray'] = None

# Numeric and plotting
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

# Deep learning
import torch

# Reproducibility
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Paths (Kaggle)
KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT   = Path('/kaggle/input')
DATA_ROOT      = KAGGLE_WORKING / 'data'
CONTAINER_DIR  = DATA_ROOT / 'container'
RUNS_DIR       = KAGGLE_WORKING / 'runs'
RESULTS_DIR    = KAGGLE_WORKING / 'results'
STATE_FILE     = KAGGLE_WORKING / 'nb2_state.json'

# Split ratios - identical to NB0/NB1
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.20, 0.10

# Roboflow pin
RF_WORKSPACE = 'dissertation-t3buk'
RF_PROJECT   = 'container-number-detection'
RF_VERSION   = 3

# Training config (plan section 5.1)
IMGSZ      = 640
EPOCHS     = 100
BATCH      = 16          # yolov8s-worldv2 fits batch 16 on T4; drop to 8 if OOM
OPTIMIZER  = 'auto'
LR0, LRF   = 0.01, 0.01
PATIENCE   = 50
DEVICE     = 0

WORLD_WEIGHTS = 'yolov8s-worldv2.pt'   # variant stated explicitly (plan 5.1)

# Eight prompts (plan section 5.2). Prompt 1 reproduces the v2 baseline.
PROMPTS = [
    'container-number',                        # 1 dataset class name (v2 baseline)
    'container number',                        # 2
    'text',                                    # 3
    'serial number',                           # 4
    'white text on shipping container',        # 5
    'identification code on cargo container',  # 6
    'letters and numbers',                     # 7
    'printed text',                            # 8
]

# Pre-registered T2 thresholds (plan section 6). Set before running; do not change after.
T2_FAIL_CEIL   = 0.10   # all prompts below this -> v2 'zero-shot fails' finding replicated
T2_RECOVER_MIN = 0.30   # any prompt above this -> v2 conclusion wrong, prompts recover the gap

DATASET_SLUG = 'ausdm-v3-container-openvocab'
KAGGLE_USERNAME_OVERRIDE = ''   # set to your username if auto-detect returns 'unknown'

for d in [DATA_ROOT, CONTAINER_DIR, RUNS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'KAGGLE_WORKING = {KAGGLE_WORKING}')
print(f'prompts        = {len(PROMPTS)}')
print(f'T2 thresholds  = fail<{T2_FAIL_CEIL}  recover>{T2_RECOVER_MIN} (mAP50-95)')
print(f'SEED           = {SEED}')

## 2. Resumable state and restore

Restore weights, metrics, and state from a previously pushed
`ausdm-v3-container-openvocab` mount so completed stages are skipped on a re-run.


In [ ]:
# _load_state / _mark_done / _is_done - stage-level idempotency across sessions
def _load_state():
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return {}

def _mark_done(stage, extra=None):
    st = _load_state()
    st[stage] = {'done': True, 'ts': time.time()}
    if extra:
        st[stage].update(extra)
    STATE_FILE.write_text(json.dumps(st, indent=2))

def _is_done(stage):
    return _load_state().get(stage, {}).get('done', False)

# _restore_from_prior_push - pull a prior NB2 output back into working
def _restore_from_prior_push():
    if STATE_FILE.exists():
        return 'local state already present'
    if not KAGGLE_INPUT.exists():
        return 'no /kaggle/input'
    for cand in KAGGLE_INPUT.rglob('nb2_state.json'):
        src = cand.parent
        shutil.copy2(cand, STATE_FILE)
        for f in ['container_openvocab_finetune_metrics.csv', 'container_prompt_sweep.csv']:
            if (src / f).exists():
                shutil.copy2(src / f, RESULTS_DIR / f)
        for sub in ['weights', 'figures']:
            if (src / sub).exists():
                shutil.copytree(src / sub, RESULTS_DIR / sub, dirs_exist_ok=True)
        return f'restored from {src}'
    return 'no prior push mounted'

print('restore:', _restore_from_prior_push())
print('current state:', json.dumps(_load_state(), indent=2))

## 3. Secret assertion and Kaggle username - fail before any compute

In [ ]:
# _get_roboflow_key - Roboflow Secret, needed to rebuild the container data
def _get_roboflow_key():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
    except Exception as e:
        raise RuntimeError(
            'ROBOFLOW_API_KEY Kaggle Secret not found. Add-ons -> Secrets -> add '
            'ROBOFLOW_API_KEY, attach, restart the kernel. Original error: %s' % e)

ROBOFLOW_API_KEY = _get_roboflow_key()
assert len(ROBOFLOW_API_KEY) > 10, 'Roboflow key looks too short'

# _kaggle_user - try every credential source Kaggle might expose, in order
def _kaggle_user():
    if KAGGLE_USERNAME_OVERRIDE:
        return KAGGLE_USERNAME_OVERRIDE
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        u = api.config_values.get('username') or api.get_config_value('username')
        if u:
            return u
    except Exception:
        pass
    u = os.environ.get('KAGGLE_USERNAME')
    if u:
        return u
    kj = Path.home() / '.kaggle' / 'kaggle.json'
    if kj.exists():
        try:
            return json.loads(kj.read_text())['username']
        except Exception:
            pass
    if KAGGLE_INPUT.exists():
        for p in KAGGLE_INPUT.rglob('*'):
            parts = p.parts
            if 'datasets' in parts:
                i = parts.index('datasets')
                if i + 1 < len(parts):
                    return parts[i + 1]
    return 'unknown'

KAGGLE_USER = _kaggle_user()
print(f'Roboflow key   : OK ({len(ROBOFLOW_API_KEY)} chars)')
print(f'Kaggle username: {KAGGLE_USER}')
if KAGGLE_USER == 'unknown':
    print('WARNING: set KAGGLE_USERNAME_OVERRIDE in section 1 before the dry-run push.')

## 4. Rebuild the container dataset (Roboflow v3, seed-0 re-split)

Deterministic, identical to NB0/NB1: the same 701 / 200 / 100 split and the same
held-out test images, so the open-vocab numbers are comparable to the supervised
baselines in NB1.


In [ ]:
# _download_container - Roboflow SDK, pinned to v3
def _download_container():
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    proj = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
    tgt  = KAGGLE_WORKING / f'rf_raw_v{RF_VERSION}'
    if not tgt.exists():
        proj.version(RF_VERSION).download('yolov8', location=str(tgt))
    return tgt

# _resplit_container - pool all 1001, shuffle with seed 0, re-partition 701/200/100
def _resplit_container(rf_tgt, out_dir):
    if out_dir.exists() and any(out_dir.iterdir()):
        shutil.rmtree(out_dir)
    for split in ['train','valid','test']:
        for sub in ['images','labels']:
            (out_dir / split / sub).mkdir(parents=True, exist_ok=True)
    items = []
    for split in ['train','valid','test']:
        for img in sorted((rf_tgt / split / 'images').iterdir()):
            lab = rf_tgt / split / 'labels' / (img.stem + '.txt')
            items.append((img, lab))
    items.sort(key=lambda t: t[0].name)
    random.Random(SEED).shuffle(items)
    n = len(items)
    n_train = int(round(n * TRAIN_FRAC)); n_val = int(round(n * VAL_FRAC))
    plan = {'train': items[:n_train],
            'valid': items[n_train:n_train+n_val],
            'test' : items[n_train+n_val:]}
    for split, pairs in plan.items():
        for img, lab in pairs:
            shutil.copy2(img, out_dir / split / 'images' / img.name)
            if lab.exists():
                shutil.copy2(lab, out_dir / split / 'labels' / lab.name)
    return {s: len(v) for s, v in plan.items()}

if not _is_done('container_data'):
    rf_tgt = _download_container()
    total = sum(len(list((rf_tgt / s / 'images').iterdir())) for s in ['train','valid','test'])
    assert total == 1001, f'Expected 1001 images, got {total}'
    counts = _resplit_container(rf_tgt, CONTAINER_DIR)
    assert counts == {'train':701,'valid':200,'test':100}, f'Unexpected split: {counts}'
    print('container re-split counts:', counts)
    _mark_done('container_data', {'counts': counts})
else:
    print('SKIP container_data, counts:', _load_state()['container_data']['counts'])

CONTAINER_YAML = CONTAINER_DIR / 'data.yaml'
CONTAINER_YAML.write_text(f'''path: {CONTAINER_DIR}
train: train/images
val:   valid/images
test:  test/images
nc: 1
names: ['container-number']
''')
print('container data.yaml:', CONTAINER_YAML,
      '| test images:', len(list((CONTAINER_DIR/'test'/'images').iterdir())))

## 5. Neutralise the raytune callback

In [ ]:
# Rewrite the callback FUNCTION to a no-op (same as NB0/NB1)
try:
    from ultralytics.utils.callbacks import raytune as _rt
    _rt.on_fit_epoch_end.__code__ = (lambda trainer: None).__code__
    _rt.callbacks = {}
    _rt.tune = None
    print('raytune callback neutralised (code swapped to no-op)')
except Exception as e:
    print('raytune patch skipped:', e)

## 6. Authenticated dry-run push

Catch owner-id / permission problems in under a minute, before the ~1.4 h fine-tune.


In [ ]:
PUSH_DIR = KAGGLE_WORKING / 'push_stage'

# _stage_push_dir - rebuild staging folder and metadata
def _stage_push_dir(files=None, dirs=None):
    if PUSH_DIR.exists():
        shutil.rmtree(PUSH_DIR)
    PUSH_DIR.mkdir(parents=True, exist_ok=True)
    for f in (files or []):
        if Path(f).exists():
            shutil.copy2(f, PUSH_DIR / Path(f).name)
    for d in (dirs or []):
        if Path(d).exists():
            shutil.copytree(d, PUSH_DIR / Path(d).name, dirs_exist_ok=True)
    meta = {'title': 'AusDM v3 container open-vocabulary',
            'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
            'licenses': [{'name': 'CC-BY-SA-4.0'}]}
    (PUSH_DIR / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2))
    return meta

# _push - create on first run, version-bump thereafter
def _push(message):
    # Robust push. The kaggle CLI can exit 0 while printing 'already in use',
    # so success is judged from the output text, not the return code. If the
    # dataset already exists, version-bump instead of create.
    def _run(sub):
        r = subprocess.run(['kaggle','datasets'] + sub, capture_output=True, text=True)
        return (r.stdout + r.stderr)
    out = _run(['create','-p',str(PUSH_DIR),'-r','zip'])
    if 'invalid owner id' in out.lower():
        print('CREATE:', out.strip()[-200:]); return False
    if 'is being created' in out.lower() or 'successfully' in out.lower():
        print('CREATE ok'); return True
    vout = _run(['version','-p',str(PUSH_DIR),'-m',message,'-r','zip'])
    print('VERSION:', vout.strip()[-220:])
    return ('is being created' in vout.lower()) or ('successfully' in vout.lower())
assert KAGGLE_USER != 'unknown', 'Set KAGGLE_USERNAME_OVERRIDE in section 1 and re-run.'

(RESULTS_DIR / 'nb2_marker.txt').write_text('nb2 dry-run ok\n')
_stage_push_dir(files=[RESULTS_DIR / 'nb2_marker.txt'])
if _push('NB2 dry-run'):
    print('dry-run push OK ->', f'{KAGGLE_USER}/{DATASET_SLUG}')
else:
    raise RuntimeError('Dry-run push failed. Fix auth / username before training.')

## 7. Fine-tune YOLO-World (100 epochs, batch 16)

`yolov8s-worldv2.pt` with the class set to `container-number`, same schedule as
the supervised baselines so the fine-tuned number is comparable to NB1.


In [ ]:
# YOLO-World fine-tune
WORLD_RUN  = RUNS_DIR / 'yoloworld_container'
WORLD_BEST = WORLD_RUN / 'weights' / 'best.pt'

if not _is_done('train_world') and not WORLD_BEST.exists():
    from ultralytics import YOLOWorld
    model = YOLOWorld(WORLD_WEIGHTS)
    model.set_classes(['container-number'])
    model.train(data=str(CONTAINER_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                optimizer=OPTIMIZER, lr0=LR0, lrf=LRF, patience=PATIENCE, seed=SEED,
                device=DEVICE, project=str(RUNS_DIR), name='yoloworld_container',
                exist_ok=True, verbose=False, plots=True)
    assert WORLD_BEST.exists(), 'YOLO-World best.pt not written'
    _mark_done('train_world', {'best': str(WORLD_BEST)})
    print('YOLO-World fine-tune done ->', WORLD_BEST)
else:
    print('SKIP train_world, best at', WORLD_BEST)

## 8. Evaluate the fine-tuned model on the held-out test split

In [ ]:
# _eval_world_test - test-split metrics for a given YOLO-World checkpoint + class
def _eval_world_test(weights, classes, name):
    from ultralytics import YOLOWorld
    m = YOLOWorld(str(weights))
    m.set_classes(classes)
    mt = m.val(data=str(CONTAINER_YAML), split='test', imgsz=IMGSZ, device=DEVICE,
               project=str(RUNS_DIR), name=name, exist_ok=True, verbose=False, plots=False)
    return {'mAP50-95': round(float(mt.box.map),   4),
            'mAP50'   : round(float(mt.box.map50), 4),
            'precision': round(float(mt.box.mp),   4),
            'recall'   : round(float(mt.box.mr),   4)}

world_ft = _eval_world_test(WORLD_BEST, ['container-number'], 'world_ft_test')
world_ft.update({'model': 'yolo-world-ft', 'split': 'test'})
print('YOLO-World fine-tuned test metrics:', json.dumps(world_ft, indent=2))
_mark_done('eval_world_ft', {'metrics': world_ft})

# checkpoint the fine-tune before the sweep
(RESULTS_DIR / 'weights').mkdir(exist_ok=True)
shutil.copy2(WORLD_BEST, RESULTS_DIR / 'weights' / 'yoloworld_ft_best.pt')
rc = WORLD_RUN / 'results.csv'
if rc.exists():
    shutil.copy2(rc, RESULTS_DIR / 'yoloworld_results.csv')
pd.DataFrame([world_ft])[['model','split','mAP50-95','mAP50','precision','recall']].to_csv(
    RESULTS_DIR / 'container_openvocab_finetune_metrics.csv', index=False)
shutil.copy2(STATE_FILE, RESULTS_DIR / 'nb2_state.json')
_stage_push_dir(dirs=[RESULTS_DIR])
print('fine-tune checkpoint push:', 'OK' if _push('NB2 fine-tune checkpoint') else 'FAILED')

**8.1 Explanation.**

- Fine-tuned YOLO-World should reach the same near-saturated level as YOLOv8s and
  RT-DETR in NB1; open-vocabulary architecture is not a handicap once trained.
- The gap to watch is the zero-shot sweep below, not this number.


## 9. Zero-shot prompt sweep on the test split

Eight prompts against the **pretrained** `yolov8s-worldv2.pt`, no fine-tuning.
Each prompt sets the model's vocabulary, then the same test-split GT boxes are
scored. Prompt 1 reproduces what v2 did.


In [ ]:
# _zero_shot_eval - pretrained YOLO-World, one prompt as the whole vocabulary
def _zero_shot_eval(model, prompt, idx):
    model.set_classes([prompt])
    mt = model.val(data=str(CONTAINER_YAML), split='test', imgsz=IMGSZ, device=DEVICE,
                   project=str(RUNS_DIR), name=f'zs_{idx}', exist_ok=True,
                   verbose=False, plots=False)
    return {'idx': idx, 'prompt': prompt,
            'mAP50-95': round(float(mt.box.map),   4),
            'mAP50'   : round(float(mt.box.map50), 4),
            'precision': round(float(mt.box.mp),   4),
            'recall'   : round(float(mt.box.mr),   4)}

if not _is_done('prompt_sweep'):
    from ultralytics import YOLOWorld
    zs_model = YOLOWorld(WORLD_WEIGHTS)     # pretrained, never fine-tuned
    sweep = []
    for i, p in enumerate(PROMPTS, start=1):
        row = _zero_shot_eval(zs_model, p, i)
        print(f'  prompt {i}: {p!r:45s} mAP50-95={row["mAP50-95"]:.4f}  mAP50={row["mAP50"]:.4f}')
        sweep.append(row)
    df_sweep = pd.DataFrame(sweep)
    df_sweep.to_csv(RESULTS_DIR / 'container_prompt_sweep.csv', index=False)
    _mark_done('prompt_sweep', {'rows': sweep})
else:
    df_sweep = pd.DataFrame(_load_state()['prompt_sweep']['rows'])
    print('SKIP prompt_sweep')

print()
print(df_sweep[['idx','prompt','mAP50-95','mAP50','precision','recall']].to_string(index=False))

## 10. T2 verdict (pre-registered)

Do not soften either comparison after seeing the numbers.
- All eight prompts below 0.10 mAP50-95: the v2 finding that zero-shot fails is replicated.
- Any prompt above 0.30 mAP50-95: the v2 conclusion is wrong and prompt engineering
  recovers most of the gap. This would be the paper's most interesting result.


In [ ]:
maxrow = df_sweep.loc[df_sweep['mAP50-95'].idxmax()]
best_map = float(maxrow['mAP50-95'])
all_below_fail = bool((df_sweep['mAP50-95'] < T2_FAIL_CEIL).all())
any_recover    = bool((df_sweep['mAP50-95'] > T2_RECOVER_MIN).any())

print(f'best prompt        : #{int(maxrow["idx"])} {maxrow["prompt"]!r}')
print(f'best mAP50-95      : {best_map:.4f}')
print(f'all 8 below {T2_FAIL_CEIL} : {all_below_fail}')
print(f'any above {T2_RECOVER_MIN}   : {any_recover}')

if any_recover:
    verdict = ('RECOVER: at least one prompt exceeds 0.30. The v2 conclusion is wrong; '
               'prompt engineering recovers most of the gap. Rewrite the Discussion around this.')
elif all_below_fail:
    verdict = ('FAIL-REPLICATED: every prompt is below 0.10. The v2 finding that zero-shot '
               'open-vocabulary detection fails on container text holds.')
else:
    verdict = ('PARTIAL: best prompt sits between 0.10 and 0.30. Zero-shot is weak but not '
               'fully failed; report the range and discuss prompt sensitivity honestly.')
print('\nT2 verdict:', verdict)
_mark_done('t2_verdict', {'best_prompt': maxrow['prompt'], 'best_map': best_map,
                          'all_below_fail': all_below_fail, 'any_recover': any_recover,
                          'verdict': verdict})

**10.1 Explanation.**

- Read the verdict against the pre-registered thresholds, not against expectation.
- Record every prompt in the paper, including a strong one. The sweep is the only
  genuinely new empirical contribution in v3 (plan 5.2), so nothing is cut.
- The fine-tuned vs zero-shot gap is the quantity the Discussion is built on.


## 11. Prediction sanity figures - boxes on test images

Mandatory box check. Shows the fine-tuned model and the best zero-shot prompt on
the same test images, so the qualitative figure contrasts the two regimes.


In [ ]:
# _draw_world - draw predicted boxes for a YOLO-World checkpoint + class on n test images
FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

def _draw_world(weights, classes, name, n=6, conf=0.25):
    from ultralytics import YOLOWorld
    m = YOLOWorld(str(weights)); m.set_classes(classes)
    imgs = sorted((CONTAINER_DIR / 'test' / 'images').iterdir())[:n]
    ndet = 0
    for img_path in imgs:
        res = m.predict(str(img_path), imgsz=IMGSZ, conf=conf, device=DEVICE, verbose=False)[0]
        im = Image.open(img_path).convert('RGB'); dr = ImageDraw.Draw(im)
        for b in res.boxes:
            x1,y1,x2,y2 = [float(v) for v in b.xyxy[0]]
            dr.rectangle([x1,y1,x2,y2], outline='lime', width=4)
            dr.text((x1, max(0,y1-12)), f'{float(b.conf[0]):.2f}', fill='lime')
            ndet += 1
        im.save(FIG_DIR / f'{name}_{img_path.stem}.png')
    return len(imgs), ndet

best_prompt = _load_state()['t2_verdict']['best_prompt']
n_ft, d_ft = _draw_world(WORLD_BEST,    ['container-number'], 'pred_world_ft')
n_zs, d_zs = _draw_world(WORLD_WEIGHTS, [best_prompt],        'pred_world_zs_best')
print(f'fine-tuned figures : {n_ft} imgs, {d_ft} boxes drawn')
print(f'zero-shot (best {best_prompt!r}): {n_zs} imgs, {d_zs} boxes drawn')

In [ ]:
# Inline preview so the box check happens in the notebook
from IPython.display import Image as IPImg, display
for p in (sorted(FIG_DIR.glob('pred_world_ft_*'))[:2] +
          sorted(FIG_DIR.glob('pred_world_zs_best_*'))[:2]):
    print(p.name); display(IPImg(str(p), width=420))

**11.1 Explanation.** Fine-tuned frames should sit tight on the container
number. The zero-shot frames may be absent or misplaced; that absence is itself
the result, and it must be visible rather than implied.


## 12. Final push - `ausdm-v3-container-openvocab`

In [ ]:
# Stage every artifact NB5 needs and push the final version
(RESULTS_DIR / 'weights').mkdir(exist_ok=True)
shutil.copy2(WORLD_BEST, RESULTS_DIR / 'weights' / 'yoloworld_ft_best.pt')
rc = WORLD_RUN / 'results.csv'
if rc.exists():
    shutil.copy2(rc, RESULTS_DIR / 'yoloworld_results.csv')
_mark_done('final_push')
shutil.copy2(STATE_FILE, RESULTS_DIR / 'nb2_state.json')

_stage_push_dir(dirs=[RESULTS_DIR])
print('final push:', 'OK' if _push('NB2 final: fine-tune + prompt sweep') else 'FAILED')

## 13. Conclusion and handoff to NB5

In [ ]:
print('NB2 state:', json.dumps(_load_state(), indent=2))
print()
print('=== fine-tuned YOLO-World (test) ===')
print(pd.read_csv(RESULTS_DIR / 'container_openvocab_finetune_metrics.csv').to_string(index=False))
print()
print('=== zero-shot prompt sweep (test) ===')
print(pd.read_csv(RESULTS_DIR / 'container_prompt_sweep.csv')[
      ['idx','prompt','mAP50-95','mAP50']].to_string(index=False))
print()
print('T2 verdict:', _load_state()['t2_verdict']['verdict'])
print('Pushed     :', f'{KAGGLE_USER}/{DATASET_SLUG}')
print('Next       : NB5 mounts this + ausdm-v3-container-supervised for Table 1 and the sweep table')

**Result.** YOLO-World is fine-tuned to the container-number class and scored on
the held-out test split, then the pretrained model is run zero-shot across eight
prompts on the same split. The **T2** verdict is decided against the
pre-registered thresholds (fail below 0.10, recover above 0.30). The fine-tune
checkpoint, fine-tune metrics, the full prompt-sweep CSV, and box-drawn figures
are pushed as `ausdm-v3-container-openvocab` for NB5.

**Why this notebook matters.** With the supervised baselines near-saturated
(NB1), the fine-tuned-versus-zero-shot gap and the prompt sensitivity are the
paper's main empirical contribution. Every prompt is reported, including any that
scores well, so the finding stands whichever way T2 resolves.
